# Implementing Tools & Tool Calling in LLMs for Agentic AI with LangChain

![](https://i.imgur.com/N0zUCi0.png)

# Demo 1 — Implementing Tools & Tool Calling for Agentic AI with LangChain


## 🛠️ Tools & Tool Calling for Agentic AI Systems

### 📋 Demo Overview

This demo introduces the **foundational building blocks** of Agentic AI systems: **Tools** and **Tool Calling**.

While a standalone LLM is excellent at reasoning and generating text, it has inherent limitations — it cannot access real-time data, query databases, perform calculations, or interact with external APIs. **Tools** bridge this gap by extending an LLM's capabilities to perform actions beyond text generation.

**Tool calling** (also known as **function calling**) is the mechanism that allows an LLM to intelligently request specific tools when it recognizes that external information or actions are needed to answer a query accurately.

In this demo, we build **HealthBuddy** — an AI assistant that combines:
- **LLM reasoning** (GPT-4o-mini)
- **Custom tools** (Web Search, PubMed Research, Doctor Recommendation)
- **RAG-based retrieval** (Vector search over doctor profiles)

The focus is on understanding how to design, register, and connect tools to an LLM, enabling it to request tool calls dynamically based on user queries.

---

### 🎯 Real-World Use Case: HealthBuddy AI Assistant

HealthBuddy is an intelligent health assistant that helps users by:

| User Need | Tool Used | How It Helps |
|-----------|-----------|--------------|
| **General Health Questions** | Web Search | Retrieves current, evidence-based health information from trusted sources |
| **Scientific Evidence** | PubMed Research | Finds peer-reviewed medical articles and clinical studies |
| **Doctor Recommendations** | RAG + Vector DB | Matches user symptoms to relevant medical specialists with availability and contact info |

**Example Queries:**
- *"What are the latest treatments for Type 2 diabetes?"* → Web Search + PubMed
- *"I have persistent migraines and dizziness — which doctor should I see?"* → Doctor Recommendation (RAG)
- *"Is intermittent fasting safe for weight loss?"* → Web Search

---

### 🏗️ Architecture Overview

![](https://drive.google.com/uc?export=view&id=1MW6am5vnM2-WbEwKgiqQ9gYChRZ70kdl)

Simplified Workflow:

```
┌──────────────────────────────────────────────────────────────┐
│                        User Query                            │
└────────────────────────┬─────────────────────────────────────┘
                         │
                         ▼
┌────────────────────────────────────────────────────────────┐
│                    LLM (GPT-4.1-mini)                      │
│                  Reasoning Engine                          │
│                                                            │
│  • Analyzes query intent                                   │
│  • Decides if tools are needed                             │
│  • Generates tool call requests (if needed)                │
│  • Synthesizes final answer from tool results              │
└────────────┬────────────────────────────┬──────────────────┘
             │                            │
             │ Tool Call Request          │ Direct Answer
             ▼                            └─────────────────────┐
┌───────────────────────────────────────────────────────────┐   |
│                        Tools Layer                        │   |
├────────────────────┬──────────────────┬───────────────────┤   |
│   search_web()     │  search_pubmed() │ recommend_doctor()│   |
│   Tavily API       │  PubMed API      │  RAG + Chroma DB  │   |
└────────────────────┴──────────────────┴───────────────────┘   |
             │                           ┌──────────────────────┘
             │ Tool Results              │
             ▼                           ▼
┌────────────────────────────────────────────────────────────┐
│                    Response to User                        │
│  (Direct LLM Response or Tool Call Raw Ouputs)             │
└────────────────────────────────────────────────────────────┘
```

---

### 🛠️ Technical Stack

| Component | Technology | Role |
|-----------|-----------|------|
| **LLM** | GPT-4.1-mini (OpenAI) | Reasoning engine that decides when and which tools to call |
| **Tool Framework** | LangChain `@tool` decorator | Defines tools with schemas that LLM can understand |
| **Web Search Tool** | Tavily Search API | Retrieves current web information on health topics |
| **Research Tool** | PubMed API | Fetches peer-reviewed scientific articles |
| **Doctor DB** | ChromaDB (vector database) | Stores doctor profiles as embeddings for semantic search |
| **Embeddings** | `text-embedding-3-small` (OpenAI) | Converts doctor profiles to 1536-dim vectors |
| **Tool Binding** | LangChain `bind_tools()` | Connects tools to LLM enabling tool call requests |

---

### 📚 What You'll Learn

By the end of this demo, you will understand:

✅ **Tool Design Principles** — How to create tools that LLMs can discover, understand, and use  
✅ **Tool Schemas** — Defining clear input/output contracts using docstrings and type hints  
✅ **RAG-Based Tools** — Building a doctor recommendation tool powered by vector search  
✅ **Tool Calling Workflow** — How LLMs generate tool call requests vs. direct answers  
✅ **Tool Call Anatomy** — Understanding tool call request structure (name, args, id)  
✅ **Manual Tool Execution** — Calling tools programmatically based on LLM requests  

**Note:** This demo focuses on **understanding tool calling mechanics**. In the next demo, we'll automate tool execution using **LangGraph ReAct Agents** that handle tool calls automatically in a loop.

---

### 💡 Key Concepts

#### Tools in Agentic AI

**Tools** are functions that extend an LLM's capabilities beyond text generation. Each tool:
- Has a clear name and description that the LLM can read
- Defines an input schema (what arguments it expects)
- Performs a specific action (API call, database query, calculation)
- Returns structured output that the LLM can reason over

#### Tool Calling Workflow

1. **User submits a query** to the LLM
2. **LLM analyzes** whether it can answer directly or needs external help
3. **If tools are needed**, LLM generates a **tool call request** (not execution!)
4. **Tool call request** contains: tool name, input arguments, call ID
5. **External system** (agent or manual code) executes the tool
6. **Tool result** is fed back to the LLM
7. **LLM synthesizes** the final answer using tool outputs

**Critical Understanding:** The LLM does **not execute tools** — it only **requests** them. Execution happens externally (either manually or via an agent framework).

---

### 🎓 Prerequisites

- Basic understanding of LLMs and prompt engineering
- Familiarity with Python functions and type hints
- Basic knowledge of vector databases (covered in Week 2)
- OpenAI API access for GPT-4.1-mini and embeddings
- Tavily API key for web search (free tier available)

---

**Let's build the key components of HealthBuddy!** 🚀


---

## 📦 Environment Setup & Dependencies

### Installation

We install all required packages for building **HealthBuddy**, our tool-enabled AI assistant.

| Package | Purpose |
|---------|---------|
| `langchain` | Core LangChain framework for tool registration and orchestration |
| `langchain-openai` | OpenAI LLM (GPT-4o-mini) and embedding model integrations |
| `langchain-community` | Community tools — Tavily web search and PubMed API wrappers |
| `xmltodict` | XML parsing utility for PubMed API responses |
| `langchain-chroma` | ChromaDB vector store for doctor profile database |

In [ ]:
# Install necessary libraries
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install xmltodict
!pip install langchain-chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

---

## 🔐 API Authentication

### Setting Up API Keys

We securely configure API keys using `getpass`, which prompts for input without displaying keys in notebook output or storing them in the cell.

> **Never hardcode API keys directly in your notebook.** Using `getpass` ensures keys stay out of version control and shared outputs.

You will need:
- **OpenAI API Key** — for LLM (GPT-4o-mini) and embeddings. Get yours at [platform.openai.com/api-keys](https://platform.openai.com/api-keys)
- **Tavily API Key** — for web search functionality. Get yours at [app.tavily.com/home](https://app.tavily.com/home)

Both keys are stored as environment variables and used throughout the notebook.

In [ ]:
import os
import getpass

# OpenAI API Key (for chat & embeddings)
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key (https://platform.openai.com/account/api-keys):\n")

# Tavily API Key (for web search)
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key (https://app.tavily.com/home):\n")

Enter your OpenAI API key (https://platform.openai.com/account/api-keys):
··········
Enter your Tavily API key (https://app.tavily.com/home):
··········


---

### 🤖 LLM & Embedder Setup

We initialise two OpenAI models:

| Model | Parameter | Value | Reason |
|---|---|---|---|
| **Chat** | `model` | `gpt-4.1-mini` | Fast, cost-efficient model well-suited for RAG answer generation |
| **Chat** | `temperature` | `0` | Deterministic outputs — critical for grounded, factual enterprise reporting |
| **Embeddings** | `model` | `text-embedding-3-small` | 1536-dim vectors, strong semantic representation at low cost |

The LLM will decide:
- When a tool is required
- Which tool to call
- How to use the tool output

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Initialize the LLM with deterministic behavior: temperature=0 ensures reproducible and stable outputs
llm = ChatOpenAI(
    model_name="gpt-4.1-mini",
    temperature=0.0,
    timeout=None,
)

embedder_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
# testing the llm
response = llm.invoke('Explain AI in 1 line')

In [ ]:
print(response.text)

AI is the simulation of human intelligence processes by machines, especially computer systems.


## Preparing Database for Doctor Recommendations Tool

In this section, we prepare the knowledge base that will power our **Doctor Recommendation Tool**.

We store doctor profiles in an external **JSON file**, which acts as the single source of truth. Each profile contains structured information such as specialization, availability, location, and contact details, along with a rich natural-language description of the doctor’s expertise.

Each doctor record includes:
- Name
- Medical specialization
- Detailed description of expertise and conditions treated
- Availability timings
- Clinic or hospital location
- Contact information

### Example Doctor Record

Below is an example of a single doctor entry from the dataset:

```json
{
  "name": "Dr. Janet Dyne",
  "specialization": "Endocrinology",
  "description": "Specializes in diagnosis and long-term management of diabetes, thyroid disorders, insulin resistance, hormonal imbalances, and metabolic syndromes.",
  "available_timings": "10:00 AM - 1:00 PM",
  "location": "City Health Clinic",
  "contact": "janet.dyne@healthclinic.com"
}
```

We then convert this dataset into a **vector database using Chroma**, where:
- Only semantic fields (specialization and description) are embedded
- Full doctor details are preserved as metadata
- Retrieval is performed using vector similarity search

This setup enables **Retrieval-Augmented Generation (RAG)** inside the tool: given a user’s health query or symptoms, we retrieve the most relevant doctors from the vector database and let an LLM reason over those results to recommend one or more suitable doctors.

In a real-world application, the JSON file could easily be replaced by a backend database (PostgreSQL, MongoDB) or an external healthcare API, without changing the retrieval or recommendation logic.

### Downloading the Doctor Dataset

We download the doctor dataset as a JSON file.  
This file serves as the **single source of truth** for all doctor information and will later be embedded into a vector database for retrieval.


In [ ]:
!gdown 1DU5DB5y_dmJKGDw9jn_Z0Gy1K9ZkXN3R

Downloading...
From: https://drive.google.com/uc?id=1DU5DB5y_dmJKGDw9jn_Z0Gy1K9ZkXN3R
To: /content/doctors_db.json
100% 3.65k/3.65k [00:00<00:00, 15.3MB/s]


### Loading the Doctor Dataset from JSON

We load the doctor dataset from the downloaded JSON file into memory.  
Each entry represents a doctor along with their specialization, availability, location, and contact details.


In [ ]:
# Load structured doctor data (name/spec/desc/etc.); single source of truth for RAG
import json

# Load doctors dataset from JSON file
with open("doctors_db.json", "r") as f:
    doctors_db_docs = json.load(f)

# Preview a few doctor records
doctors_db_docs[:2]

[{'name': 'Dr. Janet Dyne',
  'specialization': 'Endocrinology',
  'description': 'Specializes in diagnosis and long-term management of diabetes, thyroid disorders, insulin resistance, hormonal imbalances, and metabolic syndromes.',
  'available_timings': '10:00 AM - 1:00 PM',
  'location': 'City Health Clinic',
  'contact': 'janet.dyne@healthclinic.com'},
 {'name': 'Dr. Don Blake',
  'specialization': 'Cardiology',
  'description': 'Treats heart-related conditions including chest pain, hypertension, coronary artery disease, arrhythmias, and post-heart-attack care.',
  'available_timings': '2:00 PM - 5:00 PM',
  'location': 'Metro Cardiac Center',
  'contact': 'don.blake@metrocardiac.com'}]

### Converting Doctor Profiles into LangChain Documents

Each doctor profile is converted into a `Document` object:

- **page_content** contains only semantic text (specialization + description) used for embeddings
- **metadata** stores the full doctor information for safe, structured retrieval later

This separation ensures accurate semantic search while preserving factual correctness.


In [ ]:
from langchain_core.documents import Document

# Convert each doctor profile into a LangChain Document
# - page_content contains only the semantic text used for embeddings
# - metadata stores the full structured doctor information for safe retrieval
doctor_documents = []

for doctor in doctors_db_docs:
    # Only include fields relevant for semantic similarity search
    page_content = f"""
    Specialization: {doctor['specialization']}
    Description: {doctor['description']}
    """

    doctor_documents.append(
        Document(
            page_content=page_content.strip(),
            metadata=doctor  # Preserve full doctor profile as metadata
        )
    )
    # page_content=embed semantic fields only; metadata=full facts for retrieval safety

# Inspect a few generated documents
doctor_documents[:2]

[Document(metadata={'name': 'Dr. Janet Dyne', 'specialization': 'Endocrinology', 'description': 'Specializes in diagnosis and long-term management of diabetes, thyroid disorders, insulin resistance, hormonal imbalances, and metabolic syndromes.', 'available_timings': '10:00 AM - 1:00 PM', 'location': 'City Health Clinic', 'contact': 'janet.dyne@healthclinic.com'}, page_content='Specialization: Endocrinology\n    Description: Specializes in diagnosis and long-term management of diabetes, thyroid disorders, insulin resistance, hormonal imbalances, and metabolic syndromes.'),
 Document(metadata={'name': 'Dr. Don Blake', 'specialization': 'Cardiology', 'description': 'Treats heart-related conditions including chest pain, hypertension, coronary artery disease, arrhythmias, and post-heart-attack care.', 'available_timings': '2:00 PM - 5:00 PM', 'location': 'Metro Cardiac Center', 'contact': 'don.blake@metrocardiac.com'}, page_content='Specialization: Cardiology\n    Description: Treats heart

### Creating the Vector Database with Chroma

We now embed the doctor documents and store them in a **Chroma vector database**.

Key points:
- OpenAI embeddings are used for semantic representation
- Cosine similarity is used for nearest-neighbor search
- The database is persisted locally so it can be reused across sessions


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Create a Chroma vector database from the doctor documents
# - documents: semantic + metadata-rich LangChain Documents
# - collection_name: logical name for this dataset
# - cosine similarity: appropriate for embedding-based semantic search
# - persist_directory: enables reuse of the database across sessions
# Persist local DB (reuse across runs); HNSW cosine for fast semantic NN search

doctor_db = Chroma.from_documents(
    documents=doctor_documents,
    collection_name="doctor_db",
    embedding=embedder_model,
    collection_metadata={"hnsw:space": "cosine"}, # to enable index to use cosine and not euclidean (default)
    persist_directory="./doctor_database"
)

### Creating a Retriever over the Doctor Database

We create a retriever that:
- Returns up to **top 3** relevant doctors
- Applies a **similarity score threshold** to filter weak matches

This retriever will be used inside our recommendation tool.

In [ ]:
# Create a retriever interface over the vector database
# - similarity_score_threshold filters out weak semantic matches
# - k limits the maximum number of retrieved doctors
# - k=3 limits context; threshold=0.2 filters noise → relevant docs only to LLM
doctor_db_retriever = doctor_db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 3, "score_threshold": 0.2}
)

In [ ]:
# Example retrieval to validate semantic matching
doctor_db_retriever.invoke("diabetes doctor")

[Document(id='f9779a2b-c442-4d7d-9e5e-bc7b58592a65', metadata={'specialization': 'Endocrinology', 'name': 'Dr. Janet Dyne', 'available_timings': '10:00 AM - 1:00 PM', 'contact': 'janet.dyne@healthclinic.com', 'description': 'Specializes in diagnosis and long-term management of diabetes, thyroid disorders, insulin resistance, hormonal imbalances, and metabolic syndromes.', 'location': 'City Health Clinic'}, page_content='Specialization: Endocrinology\n    Description: Specializes in diagnosis and long-term management of diabetes, thyroid disorders, insulin resistance, hormonal imbalances, and metabolic syndromes.'),
 Document(id='63588a16-b725-4d9b-b4aa-8197bfb2a5c2', metadata={'name': 'Dr. Dinah Lance', 'contact': 'dinah.lance@downtownmed.com', 'available_timings': '9:00 AM - 12:00 PM', 'specialization': 'General Physician', 'description': 'Handles general medical concerns including fever, infections, fatigue, body pain, and acts as the first point of diagnosis and referral.', 'locatio

## Create Tools for AI Agent

In this section, we will define the tools that our AI Agent will use to perform specific tasks.

LangChain makes it easy to create and register tools using the `Tool` class. A tool includes:
- A name and description
- The python function to be called
- An input schema that tells the model what arguments it can use

When tools are defined properly, they help the model solve more complex problems by letting it take actions and use external data. This makes the system more useful and reliable.

### Example
```python
from langchain.tools import tool

@tool
def search_web(query: str) -> str:
    """Get live information for user queries from the web."""
    # assuming we have a google_search function implemented to search on google
    return google_search(query)
```

These tools will allow the agent to retrieve information from the external environment, as well as recommend doctors from our in-memory doctor database.

The goal is to modularize the logic for different types of tasks into reusable components that can be invoked by the LLM when needed. These include:

- A **Web Search Tool** that searches the web and downloads information from relevant websites
- A **PubMed Search Tool** that retrieves information extracted from research papers in pubmed for research-grade medical content
- A **Doctor Recommendation Tool** that finds suitable doctors based on user symptoms or needs. This tool performs **Retrieval-Augmented Generation (RAG)**:

    1. Retrieves the most relevant doctors using vector similarity search
    2. Passes the retrieved doctors to an LLM for reasoning
    3. Returns one or more suitable doctors in **structured JSON format**

The LLM is constrained to reason only over retrieved doctors to prevent hallucinations.


This tool-based setup is essential for enabling agentic behavior, where the LLM reasons through a problem, decides which action to take, and requests to call the right tools to gather more information or perform a task.


In [ ]:
import json
from typing import List, Dict, Union
from langchain.tools import tool
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper
from langchain_community.utilities.pubmed import PubMedAPIWrapper


# ══════════════════════════════════════════════════════════════════════════════
# TOOL 1: WEB SEARCH
# ══════════════════════════════════════════════════════════════════════════════

# Initialize Tavily web search client
tavily_search = TavilySearchAPIWrapper()


@tool
def search_web(query: str) -> List[str]:
    """
    Search the web for current health information and evidence-based content.

    Use this tool when:
    - User asks about general health topics, treatments, or medical conditions
    - Current or recent information is needed (research from 2025)
    - Evidence-based health guidelines are required

    Args:
        query: The health-related search query string.

    Returns:
        List of formatted strings containing title, content, and source URL from
        top web results. Returns empty list if no results found.

    Raises:
        Exception: If Tavily API call fails, returns error message in list.

    Example:
        >>> results = search_web("latest treatments for Type 2 diabetes")
        >>> print(len(results))
        5
    """
    try:
        # Perform advanced web search with full content retrieval
        results = tavily_search.raw_results(
            query=query,
            max_results=5,
            search_depth='advanced',
            include_answer=False,
            include_raw_content=True
        )

        # Extract results and filter for content availability
        docs = results.get('results', [])
        docs = [doc for doc in docs if doc.get("raw_content") is not None]

        if not docs:
            return ["No web results found for the given query."]

        # Format results with clear structure
        formatted_docs = [
            f"## Title\n{doc['title']}\n\n"
            f"## Content\n{doc['raw_content']}\n\n"
            f"## Source\n{doc['url']}"
            for doc in docs
        ]

        return formatted_docs

    except Exception as e:
        return [f"Error performing web search: {str(e)}"]


# ══════════════════════════════════════════════════════════════════════════════
# TOOL 2: PUBMED RESEARCH SEARCH
# ══════════════════════════════════════════════════════════════════════════════

# Initialize PubMed API wrapper with content limits
pubmed_search = PubMedAPIWrapper(
    top_k_results=5,
    doc_content_chars_max=5000
)


@tool
def search_pubmed(query: str) -> List[str]:
    """
    Search PubMed for peer-reviewed scientific articles and clinical studies.

    Use this tool when:
    - User asks for scientific evidence or research backing
    - Clinical studies or medical research papers are needed
    - Questions about treatment efficacy or medical mechanisms

    Args:
        query: The medical research query string.

    Returns:
        List of formatted article strings with title, authors, abstract, and
        publication info. Returns error message list if search fails.

    Raises:
        Exception: If PubMed API call fails, returns error message in list.

    Example:
        >>> articles = search_pubmed("diabetes treatment clinical trials 2024")
        >>> print(len(articles))
        5
    """
    try:
        # Query PubMed API for relevant articles
        results = pubmed_search.run(query)

        if results:
            # Split concatenated results into individual articles
            articles = results.split("\n\n")
            return articles
        else:
            return ["No scientific articles found for the given query."]

    except Exception as e:
        return [f"Error fetching PubMed articles: {str(e)}"]


# ══════════════════════════════════════════════════════════════════════════════
# TOOL 3: DOCTOR RECOMMENDATION (RAG)
# ══════════════════════════════════════════════════════════════════════════════

@tool
def recommend_doctor(query: str) -> str:
    """
    Recommend suitable doctors based on user symptoms or health conditions
    using Retrieval-Augmented Generation (RAG) over doctor profiles.

    This tool performs semantic search over a vector database of doctor profiles,
    retrieves the most relevant specialists, and uses an LLM to reason over
    the retrieved profiles to make medically appropriate recommendations.

    Use this tool when:
    - User describes symptoms and needs doctor recommendations
    - Questions like "which doctor should I see for [condition]"
    - User asks for specialist availability or contact information

    Args:
        query: User's symptom description or health condition query.

    Returns:
        JSON string containing recommended doctors with their specialization,
        expertise match explanation, availability, location, and contact info.
        Returns error dict if retrieval or LLM call fails.

    Raises:
        Exception: If vector search or LLM invocation fails, returns error dict.

    Example:
        >>> result = recommend_doctor("persistent migraines and dizziness")
        >>> data = json.loads(result)
        >>> print(data['recommended_doctors'][0]['name'])
        'Dr. Sarah Chen'
    """
    try:
        # Retrieve top 3 relevant doctor profiles using vector similarity search
        retrieved_docs = doctor_db_retriever.invoke(query)

        if not retrieved_docs:
            return json.dumps({
                "recommended_doctors": [],
                "message": "No suitable doctors found in database for this query."
            })

        # Format retrieved doctor profiles for LLM context
        doctor_context = "\n\n".join([
            f"Name: {doc.metadata['name']}\n"
            f"Specialization: {doc.metadata['specialization']}\n"
            f"Expertise: {doc.metadata['description']}\n"
            f"Available: {doc.metadata['available_timings']}\n"
            f"Location: {doc.metadata['location']}\n"
            f"Contact: {doc.metadata['contact']}"
            for doc in retrieved_docs
        ])

        # Structured prompt for LLM reasoning over retrieved doctors
        prompt = f"""You are a medical assistant helping match patients to doctors.

User query:
"{query}"

Retrieved doctor profiles from database:
{doctor_context}

Instructions:
- Recommend one or more doctors based on medical relevance to the query
- Prioritize specialists whose expertise matches the described symptoms
- Include a General Physician ONLY if symptoms are vague or no specialist is appropriate
- Do NOT make up or hallucinate doctor information
- For each recommended doctor, briefly explain why they are a good match
- Return output strictly in valid JSON format without markdown or code fences

JSON Schema:
{{
  "recommended_doctors": [
    {{
      "name": "...",
      "specialization": "...",
      "expertise_match": "Brief explanation of why this doctor is recommended",
      "location": "...",
      "available_timings": "...",
      "contact": "..."
    }}
  ]
}}"""

        # Invoke LLM for reasoning over retrieved doctors
        response = llm.invoke(prompt)

        # Return the structured recommendation
        return response.content

    except Exception as e:
        return json.dumps({
            "error": f"Error generating doctor recommendations: {str(e)}",
            "recommended_doctors": []
        })


print("✓  All 3 tools registered successfully")
print(f"  • search_web() — Web search via Tavily")
print(f"  • search_pubmed() — PubMed research articles")
print(f"  • recommend_doctor() — RAG-based doctor matching")

✓  All 3 tools registered successfully
  • search_web() — Web search via Tavily
  • search_pubmed() — PubMed research articles
  • recommend_doctor() — RAG-based doctor matching


## Testing out our Tools

You can just call the above tools manually by using the `.invoke(...)` function with the necessary inputs.

The following code shows some example queries and outputs on calling these tools

In [ ]:
results = search_web.invoke('Recent treatments in Diabetes')
print(len(results))
print(results[0][:3000])

5
## Title
A Brighter Future for Diabetes: New Advances on the Horizon

## Content
A Brighter Future for Diabetes: New Advances on the Horizon | AdventHealth News and Stories
[Skip to main content](https://www.adventhealth.com/news/a-brighter-future-diabetes-new-advances-horizon#main-content)

[![Image 1: AdventHealth News and Stories](https://www.adventhealth.com/sites/default/files/media/adventhealth_news_stories_4c.svg)](https://www.adventhealth.com/news "AdventHealth News and Stories")

 Search  

Menu

Close
AdventHealth News and Stories

 Search  

*   [News and Stories](https://www.adventhealth.com/news)
*   [Care](https://www.adventhealth.com/news/care)
*   [Community](https://www.adventhealth.com/news/community)
*   [People](https://www.adventhealth.com/news/people)
*   [Perspectives](https://www.adventhealth.com/news/perspectives)
*   [Media Resources](https://www.adventhealth.com/news/media-resources-and-contacts)

[AdventHealth Home](https://www.adventhealth.com/)

*   [New

In [ ]:
results = search_pubmed.invoke('Recent treatments in Diabetes')
print(len(results))
print(results[0][:3000])

Too Many Requests, waiting for 0.20 seconds...
3
Published: 2026-02-21
Title: Artificial cycle frozen embryo transfer improves uterine perfusion during later pregnancy but is associated with higher preeclampsia incidence.
Copyright Information: Copyright © 2026. Published by Elsevier Inc.
Summary::
BACKGROUND: Uterine artery pulsatility index (UtAPI) is a key biomarker for preeclampsia (PE) screening and the most reliable indicator of uterine perfusion across all pregnancy trimesters. Although recent findings reveal a significant decrease in UtAPI during first trimester in AC-FET( Artificial Cycle-Frozen Embryo Transfer) pregnancies, no previous study evaluated whether this decrease persists throughout the second and third trimesters, when hormonal treatment is discontinued. Considering the crucial role of UtAPI during the second half of pregnancy risk assessment, recommended by international guidelines to ensure early PE detection and proper pregnancy monitoring, we set out to perform

In [ ]:
result = recommend_doctor.invoke('Treatments for Diabetes')
print(result)

{
  "recommended_doctors": [
    {
      "name": "Dr. Janet Dyne",
      "specialization": "Endocrinology",
      "expertise_match": "Specializes in diagnosis and long-term management of diabetes, making her the most relevant specialist for treatments related to diabetes.",
      "location": "City Health Clinic",
      "available_timings": "10:00 AM - 1:00 PM",
      "contact": "janet.dyne@healthclinic.com"
    }
  ]
}


## Explore LLM tool calling with custom tools

An agent is basically an LLM which has the capability to automatically call relevant functions to perform complex or tool-based tasks based on input human prompts.

Tool calling also popularly known as function calling is the ability to reliably enable such LLMs to call external tools and APIs.

We will leverate the custom tools we created earlier in the previous section and try to see if the LLM can automatically call the right tools based on input prompts

## Tool calling for LLMs

Tool calling allows a model to respond to a given prompt by generating output that matches a user-defined schema. While the name implies that the model is performing some action, this is actually not the case! The model is coming up with the arguments to a tool, and actually running the tool (or not) is up to the user or agent defined by the user.

Many LLM providers, including Anthropic, Cohere, Google, Mistral, OpenAI, and others, support variants of a tool calling feature. These features typically allow requests to the LLM to include available tools and their schemas, and for responses to include calls to these tools.

For example, if a user asks a question that requires a search or data lookup, the model can decide to call a specific tool with a tool call request.

Tool calling in LangChain works by:
1. Registering defined tool functions using `@tool` decorator
2. Binding the tools to the model using `llm.bind_tools([tool1, tool2, ...])`
3. Passing a user query to the bound model
4. Letting the model decide whether to use a tool or respond directly

This setup lets the model behave more like an agent that can take actions, observe results, and continue the conversation intelligently.

By using `bind_tools`, we give the LLM the ability to understand what tools are available and make requests to use them only when needed.

### Example
```python
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(model="gpt-4.1-mini")
llm_with_tools = llm.bind_tools([search_web])

response = llm_with_tools.invoke("treatments for diabetes?")
```

### Tool Call Requests

LLMs will not call and execute the tools for you directly but will request tool calls based on reasoning on the input user query if they feel that they do not have enough information to answer the question directly.

To experiment with this, let's first define our tools and bind them to our LLM

In [ ]:
# List of all tools that the LLM should be aware of
# These tools were defined earlier using the @tool decorator
tools = [search_web, search_pubmed, recommend_doctor]

# Bind the tools to the LLM so it can request tool calls when needed
# This enables tool calling functionality using LangChain
llm_with_tools = llm.bind_tools(tools=tools)

In [ ]:
# bind_tools() → LLM knows schemas/when-to-use; .invoke() yields content or tool_calls
# injected into the overall context / prompt when it goes to the LLM
llm_with_tools

RunnableBinding(bound=ChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x7b0cf0a10260>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7b0cf0a8ef90>, root_client=<openai.OpenAI object at 0x7b0d13af3ad0>, root_async_client=<openai.AsyncOpenAI object at 0x7b0cf0a9dc10>, model_name='gpt-4.1-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True), kwargs={'tools': [{'type': 'function', 'function': {'name': 'search_web', 'descript

In [ ]:
llm_with_tools.kwargs['tools']

[{'type': 'function',
  'function': {'name': 'search_web',
   'description': 'Search the web for current health information and evidence-based content.\n\nUse this tool when:\n- User asks about general health topics, treatments, or medical conditions\n- Current or recent information is needed (research from 2025)\n- Evidence-based health guidelines are required\n\nArgs:\n    query: The health-related search query string.\n\nReturns:\n    List of formatted strings containing title, content, and source URL from\n    top web results. Returns empty list if no results found.\n\nRaises:\n    Exception: If Tavily API call fails, returns error message in list.\n\nExample:\n    >>> results = search_web("latest treatments for Type 2 diabetes")\n    >>> print(len(results))\n    5',
   'parameters': {'properties': {'query': {'type': 'string'}},
    'required': ['query'],
    'type': 'object'}}},
 {'type': 'function',
  'function': {'name': 'search_pubmed',
   'description': 'Search PubMed for peer

Now we can test out these prompts using out LLM with tools.

Observe that:

- If the LLM can answer the question directly, the response will be present in `result.content`
- If the LLM can't answer the question directly, the response will be empty but will contain one or more tool call requests in `result.tool_calls`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TEST QUERIES: TOOL CALLING BEHAVIOR
# ══════════════════════════════════════════════════════════════════════════════

# Define test queries that trigger different tool calling behaviors
prompts = [
    "recent treatments available for diabetes",      # → should call search_web (current info)
    "Research papers on diabetes treatments",        # → should call search_pubmed (scientific)
    "What doctor could I visit for diabetes",        # → should call recommend_doctor (RAG)
    "Explain what is diabetes in simple terms"       # → direct answer (LLM knowledge)
]

# Store results for analysis
results = []

# System prompt with clear tool-calling instructions
sys_prompt = (
    'system',
    """Rules:
- If a query requires recent, specialized, or external information, call appropriate tools.
- If a query involves recommending a doctor, call appropriate tools.
- Only answer directly if you know the answer.
Current year is 2025.
"""
)

print("="*80)
print("  TESTING TOOL CALLING BEHAVIOR WITH LLM")
print("="*80)
print(f"\n  Testing {len(prompts)} queries to observe when LLM calls tools vs. direct answers\n")

# Batch test: LLM reasons about each query and decides tool usage
for i, prompt in enumerate(prompts, 1):
    # Create user message
    user_prompt = ('human', prompt)

    print(f"\n[Query {i}/{len(prompts)}] {prompt}")
    print("-" * 80)

    # Combine system instructions with user query
    final_prompt = [sys_prompt, user_prompt]

    # Invoke LLM with tools bound
    result = llm_with_tools.invoke(final_prompt)
    results.append(result)

    # Display LLM decision
    # If the model decided that a tool should be called
    if result.tool_calls:
        print(f"  ✓  LLM Decision: CALL TOOLS ({len(result.tool_calls)} tool(s) requested)")
        for tc in result.tool_calls:
            print(f"      • Tool: {tc['name']}")
            print(f"        Args: {tc['args']}")
    # If the model gave a direct response without needing a tool
    else:
        print(f"  ✓  LLM Decision: DIRECT ANSWER")
        print(f"      Response: {result.content[:100]}...")

print("\n" + "="*80)
print(f"  ✓  Completed {len(results)} test queries")
print("="*80)

  TESTING TOOL CALLING BEHAVIOR WITH LLM

  Testing 4 queries to observe when LLM calls tools vs. direct answers


[Query 1/4] recent treatments available for diabetes
--------------------------------------------------------------------------------
  ✓  LLM Decision: CALL TOOLS (1 tool(s) requested)
      • Tool: search_web
        Args: {'query': 'recent treatments available for diabetes 2025'}

[Query 2/4] Research papers on diabetes treatments
--------------------------------------------------------------------------------
  ✓  LLM Decision: CALL TOOLS (1 tool(s) requested)
      • Tool: search_pubmed
        Args: {'query': 'diabetes treatments'}

[Query 3/4] What doctor could I visit for diabetes
--------------------------------------------------------------------------------
  ✓  LLM Decision: CALL TOOLS (1 tool(s) requested)
      • Tool: recommend_doctor
        Args: {'query': 'diabetes'}

[Query 4/4] Explain what is diabetes in simple terms
-----------------------------------

### Tool Call Responses with Manual Tool Calling

While an Agentic Framework like LangChain or LangGraph can leverage AI Agents themselves to handle tool call requests by calling them automatically and getting tool call results (to further feed it to the LLM for more reasoning), here we will call these tool call requests manually just to show how the results would look like.

In future demos the AI Agents will handle this automatically.

We start off by creating a mapping between the tool call names (strings) and the actual tool functions (python function names)

In [ ]:
# create a mapping between tool call request name and the actual tool call function
tool_mapper = {
    "search_web": search_web,
    "search_pubmed": search_pubmed,
    "recommend_doctor": recommend_doctor
}

If you inspect the tool call requests for any input prompt,
we have the tool name and the input arguments which will need to be extracted and then we will need to call the actual tool with the input arguments

For example based on the following tool call request, we would need to call:

`search_web.invoke(query='treatments available for diabetes')`

In [ ]:
# check out a sample tool call request
results[0].tool_calls

[{'name': 'search_web',
  'args': {'query': 'recent treatments available for diabetes 2025'},
  'id': 'call_juszLrlEftFIINNMuD0zVZV2',
  'type': 'tool_call'}]

We can easily automate this by extracting the tool call request `name` and `args` and invoking the actual python tool function matching the `name` with the input arguments (`args`)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MANUAL TOOL EXECUTION
# ══════════════════════════════════════════════════════════════════════════════

print("="*80)
print("  EXECUTING TOOL CALLS MANUALLY")
print("="*80)
print("\n  Processing all tool call requests from LLM responses...\n")

# Iterate through each LLM response
for result_idx, result in enumerate(results, 1):
    tool_call_requests = result.tool_calls

    # Skip if no tools were requested for this query
    if not tool_call_requests:
        print(f"[Result {result_idx}] No tools called - direct answer provided")
        print(f"      Response: {result.content}")
        print("-" * 80)
        continue

    print(f"[Result {result_idx}] Executing {len(tool_call_requests)} tool call(s)")
    print("-" * 80)

    # Execute each requested tool
    for tc_idx, tool_call in enumerate(tool_call_requests, 1):
        # Extract tool information from LLM request
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        # Map tool name to actual function
        selected_tool = tool_mapper[tool_name]

        # Execute the tool with LLM-provided arguments
        print(f"\n  [{tc_idx}] Executing Tool: {tool_name}")
        print(f"      Arguments: {tool_args}")
        print(f"      Status: Running...")

        try:
            tool_output = selected_tool.invoke(tool_args)
            print(f"      Status: ✓ Success")
            print(f"\n  Tool Output Preview:")
            print(f"  {'-' * 76}")
            print(f"  {tool_output}")
            print(f"  {'-' * 76}")

        except Exception as e:
            print(f"      Status: ✗ Failed")
            print(f"      Error: {str(e)}")

        print()  # Blank line between tool executions

    print("="*80 + "\n")

print("\n" + "="*80)
print("  ✓  All tool executions completed")
print("="*80)

  EXECUTING TOOL CALLS MANUALLY

  Processing all tool call requests from LLM responses...

[Result 1] Executing 1 tool call(s)
--------------------------------------------------------------------------------

  [1] Executing Tool: search_web
      Arguments: {'query': 'recent treatments available for diabetes 2025'}
      Status: Running...
      Status: ✓ Success

  Tool Output Preview:
  ----------------------------------------------------------------------------
  ['## Title\nDiabetes treatment in 2025: can scientific advances keep pace with ...\n\n## Content\nDiabetes treatment in 2025: can scientific advances keep pace with prevalence? - PMC\n===============\n[Skip to main content](https://pmc.ncbi.nlm.nih.gov/articles/PMC3498849/#main-content)\n\n![Image 1](https://cdn.ncbi.nlm.nih.gov/pmc/pd-medc-pmc-cloudpmc-viewer/production/8a2a0396/var/data/static/img/us_flag.svg)\n\nAn official website of the United States government\n\nHere\'s how you know\n\nHere\'s how you know\n\n![Ima

In the upcoming hands-on demo you will see how to connect LLMs, tools and instruction prompts together to automate this and build your first Agentic AI System with Tool-Use!